# Chapter 12: Ethical Reasoning Agent

**Book:** *30 Agents Every AI Engineer Must Build*
**Author:** Imran Ahmad — Packt Publishing
**Chapter:** 12 — Ethical and Explainable Agents (p.3–23)

---

This notebook implements the **Ethical Reasoning Agent** architecture and the
**HR Assistant with Fairness Constraints** case study. Topics covered:

- Value Alignment Frameworks with deontic logic (p.5–7)
- The Ethical Consistency Theorem (p.7)
- EthicalReasoningAgent with modular validators (p.8–9)
- EU AI Act compliance checking (p.10–11)
- Impossibility of Fairness Theorem (p.12–13)
- Bias detection: demographic parity, equal opportunity, disparate impact (p.14–19)
- Bias monitoring pipeline with streaming alerts (p.17–19)
- Fair hiring pipeline: anonymize → score → detect → mitigate (p.20–23)

> **Simulation Mode:** This notebook runs fully without an API key. All LLM calls
> are handled by `MockLLM` with chapter-derived, deterministic responses.


In [ ]:
# ============================================================================
# Cell 2: Setup — Imports, sys.path, and Mode Detection
# Chapter 12, p.2 (Tech Requirements)
# Author: Imran Ahmad
# ============================================================================

import sys
import os
from pathlib import Path

# Ensure the project root is on sys.path (handles notebook subdirectory)
project_root = str(Path.cwd().parent) if Path.cwd().name == "notebooks" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Matplotlib inline for notebook plots
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Core imports from src/
from src.utils import ColorLogger, graceful_fallback, resolve_api_key, get_mode, is_simulation
from src.mock_llm import MockLLM, get_mock_llm
from src.synthetic_data import generate_hr_dataset
from src.ethical_core import (
    DeonticOperator,
    EthicalReasoningAgent,
    EUCompliantAgent,
    DemographicParityMetric,
    EqualOpportunityMetric,
    DisparateImpactMetric,
    BiasDetector,
    BiasMonitoringPipeline,
    ResumeAnalyzer,
    FairnessEnforcer,
    FairHiringAgent,
    demonstrate_impossibility_theorem,
)

# Initialize logger and resolve mode
logger = ColorLogger(name="Notebook01")

# Resolve API key (interactive prompt in notebooks)
api_key = resolve_api_key(interactive=True)
mode = get_mode()

logger.info(f"Notebook 01 ready. Mode: {mode}")
logger.info(f"NumPy: {np.__version__}, Pandas: {pd.__version__}")


## Value Alignment Frameworks — Deontic Logic (p.5–7)

Deontic logic formalizes ethical reasoning through three operators:

| Operator | Symbol | Meaning |
|---|---|---|
| **Obligation** | O(A) | Action A *must* be performed |
| **Permission** | P(A) | Action A *may* be performed |
| **Prohibition** | F(A) | Action A *must not* be performed |

**Three Axioms:**
1. **O(A) → P(A)** — If an action is obligatory, it is also permitted.
2. **F(A) ↔ ¬P(A)** — An action is forbidden if and only if it is not permitted.
3. **O(A) → ¬F(A)** — An obligatory action cannot be forbidden.


In [ ]:
# ============================================================================
# Cell 4: Deontic Logic — Three Axioms Interactive Demo
# Chapter 12, p.5–7 (Value Alignment Frameworks)
# Author: Imran Ahmad
# ============================================================================

deontic = DeonticOperator()

# Define an ethical framework for our HR agent
logger.info("Building ethical framework for HR Agent...")

# Obligations — things the agent MUST do
deontic.add_obligation("provide_explanation")
deontic.add_obligation("log_all_decisions")
deontic.add_obligation("check_bias_before_decision")

# Permissions — things the agent MAY do
deontic.add_permission("access_anonymized_data")
deontic.add_permission("request_additional_info")

# Prohibitions — things the agent MUST NOT do
deontic.add_prohibition("share_medical_details")
deontic.add_prohibition("discriminate_by_gender")
deontic.add_prohibition("bypass_consent")
deontic.add_prohibition("disable_audit")

# Display current status
status = deontic.get_status()
logger.success(f"Obligations: {status['obligations']}")
logger.success(f"Permissions: {status['permissions']}")
logger.success(f"Prohibitions: {status['prohibitions']}")

# Demonstrate Axiom 1: O(A) → P(A)
logger.info("\n--- Axiom 1: O(A) → P(A) ---")
logger.info(f"'provide_explanation' is obligatory: {deontic.is_obligatory('provide_explanation')}")
logger.info(f"'provide_explanation' is also permitted: {deontic.is_permitted('provide_explanation')}")

# Demonstrate Axiom 2: F(A) ↔ ¬P(A)
logger.info("\n--- Axiom 2: F(A) ↔ ¬P(A) ---")
logger.info(f"'share_medical_details' is forbidden: {deontic.is_forbidden('share_medical_details')}")
logger.info(f"'share_medical_details' is NOT permitted: {not deontic.is_permitted('share_medical_details')}")

# Demonstrate Axiom 3: O(A) → ¬F(A)
logger.info("\n--- Axiom 3: O(A) → ¬F(A) ---")
logger.info(f"'log_all_decisions' is obligatory: {deontic.is_obligatory('log_all_decisions')}")
logger.info(f"'log_all_decisions' is NOT forbidden: {not deontic.is_forbidden('log_all_decisions')}")


## Ethical Consistency Theorem (p.7)

**Theorem:** An action set S is *consistent* if and only if no action in S is
both obligatory and forbidden.

Formally: **∀a ∈ S: ¬(O(a) ∧ F(a))**

If a conflict is detected, the system must halt and escalate to a human reviewer.


In [ ]:
# ============================================================================
# Cell 6: Ethical Consistency Theorem — Worked Examples
# Chapter 12, p.7 (Ethical Consistency Theorem)
# Author: Imran Ahmad
# ============================================================================

# Example 1: Consistent framework (no conflicts)
logger.info("=== Example 1: Consistent Framework ===")
consistency = deontic.check_consistency()
logger.info(f"Result: {consistency['details']}")

# Example 2: Introduce a conflict — make 'log_all_decisions' both O and F
logger.info("\n=== Example 2: Introducing a Conflict ===")
conflicted = DeonticOperator()
conflicted.add_obligation("log_all_decisions")
# Manually force a conflict (bypassing the axiom safeguards for demonstration)
conflicted._prohibitions.add("log_all_decisions")

conflict_result = conflicted.check_consistency()
logger.info(f"Conflicts found: {conflict_result['conflicts']}")
logger.info(f"Result: {conflict_result['details']}")

# Example 3: Resolve the conflict
logger.info("\n=== Example 3: Resolving the Conflict ===")
conflicted._prohibitions.discard("log_all_decisions")
resolved = conflicted.check_consistency()
logger.info(f"After resolution: {resolved['details']}")


## EthicalReasoningAgent (p.8–9)

The `EthicalReasoningAgent` evaluates proposed actions against a set of modular
validators. Each validator maps a keyword pattern to a violation category.

**Architecture:**
1. **Modular validators** — plug-in rules for different domains (privacy, safety, fairness).
2. **`evaluate_action()`** — runs all validators, returns a structured compliance report.
3. **`mitigate()`** — suggests corrective actions for detected violations.
4. **`escalate_to_human()`** — flags critical issues for human review.

All methods are wrapped in `@graceful_fallback` for resilience.


In [ ]:
# ============================================================================
# Cell 8: EthicalReasoningAgent — Action Evaluation
# Chapter 12, p.8–9 (EthicalReasoningAgent)
# Author: Imran Ahmad
# ============================================================================

agent = EthicalReasoningAgent()

# Test 1: A compliant action
logger.info("=== Test 1: Compliant Action ===")
result = agent.evaluate_action(
    "Provide anonymized hiring statistics to the team lead",
    context={"user_role": "hr_manager"}
)
logger.info(f"Compliant: {result['is_compliant']}, Severity: {result['severity']}")

# Test 2: A single violation
logger.info("\n=== Test 2: Single Violation ===")
result = agent.evaluate_action(
    "Share medical details of candidate C-0042 with the hiring manager",
    context={"user_role": "hr_coordinator"}
)
logger.info(f"Compliant: {result['is_compliant']}, Violations: {len(result['violations'])}")
for cat, desc in result["violations"]:
    logger.error(f"  [{cat.upper()}] {desc}")

# Test 3: Multiple violations (critical)
logger.info("\n=== Test 3: Multiple Violations (Critical) ===")
result = agent.evaluate_action(
    "Bypass consent and share medical details via external email, then disable audit",
    context={"user_role": "unknown"}
)
logger.info(f"Compliant: {result['is_compliant']}, Severity: {result['severity']}")
for cat, desc in result["violations"]:
    logger.error(f"  [{cat.upper()}] {desc}")


In [ ]:
# ============================================================================
# Cell 9: EthicalReasoningAgent — Mitigation & Audit Trail
# Chapter 12, p.9 (Mitigation and Escalation)
# Author: Imran Ahmad
# ============================================================================

# Generate mitigations for the violations from Test 3
last_result = agent.evaluate_action(
    "Share medical details via external email",
    context={"user_role": "contractor"}
)
mitigation = agent.mitigate(last_result["violations"])

logger.info("=== Suggested Mitigations ===")
for m in mitigation["mitigations"]:
    logger.success(f"  {m}")

# Escalation for critical issues
if last_result["severity"] in ("HIGH", "CRITICAL"):
    escalation = agent.escalate_to_human(
        "Share medical details via external email",
        last_result["violations"]
    )
    logger.info(f"Escalated: {escalation['escalated']}")
    logger.info(f"Recommended reviewer: {escalation['recommended_reviewer']}")

# View full audit log
logger.info("\n=== Audit Trail ===")
audit = agent.get_audit_log()
logger.info(f"Total audit entries: {len(audit)}")
for entry in audit[-2:]:  # Show last 2
    logger.debug(
        f"  [{entry['timestamp'][:19]}] Action: '{entry['action'][:40]}...' "
        f"→ {entry['result']['severity']}"
    )


In [ ]:
# ============================================================================
# Cell 10: EU AI Act Compliance Check — Seven Requirements
# Chapter 12, p.10–11 (EUCompliantAgent)
# Author: Imran Ahmad
# ============================================================================

eu_agent = EUCompliantAgent()

# Define our system's configuration (simulate a partially-compliant system)
system_config = {
    # Requirement 1: Human Oversight — PASS
    "human_override": True,
    "escalation_path": True,
    "manual_review": True,
    # Requirement 2: Robustness — PASS
    "error_handling": True,
    "fallback_mode": True,
    "input_validation": True,
    # Requirement 3: Privacy — FAIL (missing data minimization)
    "data_encryption": True,
    "consent_management": True,
    "data_minimization": False,  # ← Violation
    # Requirement 4: Transparency — PASS
    "explanation_interface": True,
    "audit_trail": True,
    "model_documentation": True,
    # Requirement 5: Fairness — PASS
    "bias_testing": True,
    "fairness_metrics": True,
    "representative_data": True,
    # Requirement 6: Well-being — FAIL (missing stakeholder consultation)
    "impact_assessment": True,
    "stakeholder_consultation": False,  # ← Violation
    # Requirement 7: Accountability — PASS
    "responsible_party": True,
    "audit_schedule": True,
    "incident_response": True,
}

report = eu_agent.compliance_check(system_config)

logger.info(f"\n{report['summary']}")
logger.info("\n=== Per-Requirement Results ===")
for req in report["report"]:
    status_icon = "✅" if req["status"] == "PASS" else "❌"
    logger.info(f"  {status_icon} [{req['requirement_id']}] {req['requirement_name']}: {req['status']}")
    if req["status"] == "FAIL":
        failed_fields = [f for f, v in req["field_results"].items() if not v]
        logger.error(f"      Missing: {', '.join(failed_fields)}")


## Impossibility of Fairness Theorem (p.12–13)

**Key Insight:** When base rates differ across groups, it is mathematically
impossible to simultaneously satisfy all three fairness criteria:

| Regime | Satisfies | Violates |
|---|---|---|
| Demographic Parity Priority | Equal selection rates | Equal opportunity, Predictive parity |
| Equal Opportunity Priority | Equal true-positive rates | Demographic parity, Predictive parity |
| Predictive Parity Priority | Equal PPV | Demographic parity, Equal opportunity |

This is the core tradeoff that organizations must navigate. See **Table 12.1** (p.12).


In [ ]:
# ============================================================================
# Cell 12: Impossibility Theorem — Table 12.1 Visualization
# Chapter 12, p.12–13 (Impossibility of Fairness Theorem)
# Author: Imran Ahmad
# ============================================================================

theorem = demonstrate_impossibility_theorem()

logger.info(f"Theorem: {theorem['theorem']}")
logger.info(f"Reference: {theorem['reference']}")

# Visualize as a heatmap
regimes = []
criteria = ["demographic_parity", "equal_opportunity", "predictive_parity"]

for key, regime in theorem["regimes"].items():
    row = {}
    for c in criteria:
        if c in regime["satisfied"]:
            row[c] = 1  # Satisfied
        else:
            row[c] = 0  # Violated
    row["regime"] = regime["name"]
    regimes.append(row)

regime_df = pd.DataFrame(regimes).set_index("regime")

fig, ax = plt.subplots(figsize=(8, 3.5))
sns.heatmap(
    regime_df, annot=True, cmap=["#FF6B6B", "#51CF66"],
    cbar=False, linewidths=2, linecolor="white",
    fmt="d", ax=ax,
    xticklabels=["Demographic\nParity", "Equal\nOpportunity", "Predictive\nParity"],
)
ax.set_title("Impossibility of Fairness Theorem — Table 12.1 (p.12-13)\n"
             "Green = Satisfied, Red = Violated", fontsize=12, fontweight="bold")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

logger.info(f"\nConclusion: {theorem['conclusion']}")


## Bias Detection (p.14–19)

We now apply three fairness metrics to a synthetic HR dataset with **injected
gender bias** (see p.15, p.20). The dataset has 200 candidates where female
candidates receive a systematic scoring penalty, producing a four-fifths
violation (disparate impact ≈ 0.73).

**Three metrics:**
1. **Demographic Parity** — Are selection rates equal across groups?
2. **Equal Opportunity** — Are true-positive rates equal across groups?
3. **Disparate Impact** — Does the selection rate ratio meet the four-fifths rule (≥ 0.80)?


In [ ]:
# ============================================================================
# Cell 14: Generate and Explore the Synthetic HR Dataset
# Chapter 12, p.14–15 (Bias in AI — HR Context)
# Author: Imran Ahmad
# ============================================================================

hr_df = generate_hr_dataset(n=200, seed=42)

logger.info(f"Dataset shape: {hr_df.shape}")
logger.info(f"Columns: {list(hr_df.columns)}")
logger.info(f"\nGender distribution:")
for gender, count in hr_df["gender"].value_counts().items():
    logger.info(f"  {gender}: {count}")

logger.info(f"\nQualification rate by gender:")
for gender, group in hr_df.groupby("gender"):
    rate = group["qualified"].mean()
    logger.info(f"  {gender}: {rate:.1%}")

# Score distribution by gender
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
for gender in ["male", "female", "non_binary"]:
    subset = hr_df[hr_df["gender"] == gender]["raw_score"]
    axes[0].hist(subset, bins=15, alpha=0.6, label=gender)
axes[0].set_xlabel("Raw Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Score Distribution by Gender")
axes[0].axvline(x=0.65, color="red", linestyle="--", label="Threshold (0.65)")
axes[0].legend()

# Selection rates
rates = hr_df.groupby("gender")["qualified"].mean()
colors = ["#4C72B0", "#DD8452", "#55A868"]
bars = axes[1].bar(rates.index, rates.values, color=colors[:len(rates)])
axes[1].axhline(y=rates.max() * 0.8, color="red", linestyle="--",
                label=f"Four-fifths threshold ({rates.max() * 0.8:.2f})")
axes[1].set_ylabel("Selection Rate")
axes[1].set_title("Selection Rate by Gender (Four-Fifths Rule)")
axes[1].legend()

# Add value labels on bars
for bar, val in zip(bars, rates.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{val:.1%}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
# Cell 15: BiasDetector — Three Fairness Metrics
# Chapter 12, p.16–17 (Bias Detection Metrics)
# Author: Imran Ahmad
# ============================================================================

detector = BiasDetector()
analysis = detector.analyze(hr_df, group_col="gender", outcome_col="qualified")

logger.info("=== Bias Analysis Report ===")
logger.info(f"Summary: {analysis['summary']}")
logger.info(f"Severity: {analysis['severity']}")

# Demographic Parity
dp = analysis["metrics"]["demographic_parity"]
logger.info(f"\n--- Demographic Parity ---")
logger.info(f"  Group rates: {dp['group_rates']}")
logger.info(f"  Max gap: {dp['max_gap']:.4f}")

# Equal Opportunity
eo = analysis["metrics"].get("equal_opportunity", {})
if eo:
    logger.info(f"\n--- Equal Opportunity ---")
    logger.info(f"  Group TPR: {eo['group_tpr']}")
    logger.info(f"  Max gap: {eo['max_gap']:.4f}")

# Disparate Impact
di = analysis["metrics"]["disparate_impact"]
logger.info(f"\n--- Disparate Impact ---")
logger.info(f"  Reference group: {di['reference_group']}")
logger.info(f"  Impact ratios: {di['impact_ratios']}")
logger.info(f"  Min ratio: {di['min_ratio']:.4f}")
logger.info(f"  Four-fifths violation: {di['four_fifths_violation']}")

# Recommendations
if analysis["recommendations"]:
    logger.info("\n--- Recommendations ---")
    for rec in analysis["recommendations"]:
        logger.error(f"  → {rec}")


In [ ]:
# ============================================================================
# Cell 16: Bias Metric Dashboard — Visual Summary
# Chapter 12, p.16–17 (Bias Detection Visualization)
# Author: Imran Ahmad
# ============================================================================

di = analysis["metrics"]["disparate_impact"]
dp = analysis["metrics"]["demographic_parity"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Disparate Impact Ratios
groups = list(di["impact_ratios"].keys())
ratios = [di["impact_ratios"][g] for g in groups]
colors_di = ["#51CF66" if r >= 0.80 else "#FF6B6B" for r in ratios]
axes[0].barh(groups, ratios, color=colors_di)
axes[0].axvline(x=0.80, color="red", linestyle="--", linewidth=2, label="Four-fifths (0.80)")
axes[0].set_xlabel("Disparate Impact Ratio")
axes[0].set_title("Disparate Impact by Group")
axes[0].legend()
for i, v in enumerate(ratios):
    axes[0].text(v + 0.02, i, f"{v:.2f}", va="center", fontweight="bold")

# Plot 2: Demographic Parity Rates
dp_groups = list(dp["group_rates"].keys())
dp_rates = [dp["group_rates"][g] for g in dp_groups]
axes[1].bar(dp_groups, dp_rates, color=["#4C72B0", "#DD8452", "#55A868"][:len(dp_groups)])
axes[1].set_ylabel("Selection Rate")
axes[1].set_title("Demographic Parity")
for i, v in enumerate(dp_rates):
    axes[1].text(i, v + 0.01, f"{v:.1%}", ha="center", fontweight="bold")

# Plot 3: Severity Gauge
severity_map = {"LOW": (0.2, "#51CF66"), "HIGH": (0.6, "#FFD43B"), "CRITICAL": (0.9, "#FF6B6B")}
sev_val, sev_color = severity_map.get(analysis["severity"], (0.5, "gray"))
axes[2].barh(["Severity"], [sev_val], color=sev_color, height=0.4)
axes[2].set_xlim(0, 1)
axes[2].set_title(f"Bias Severity: {analysis['severity']}")
axes[2].set_xlabel("Risk Level")
axes[2].axvline(x=0.33, color="gray", linestyle=":", alpha=0.5)
axes[2].axvline(x=0.66, color="gray", linestyle=":", alpha=0.5)
axes[2].text(0.15, 0, "LOW", ha="center", va="bottom", fontsize=9, color="gray")
axes[2].text(0.50, 0, "HIGH", ha="center", va="bottom", fontsize=9, color="gray")
axes[2].text(0.83, 0, "CRITICAL", ha="center", va="bottom", fontsize=9, color="gray")

plt.suptitle("Bias Detection Dashboard (Chapter 12, p.16-17)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## Bias Monitoring Pipeline (p.17–19)

In production, bias must be monitored **continuously** — not just at training time.
The `BiasMonitoringPipeline` implements:

1. A **sliding window** of recent decisions.
2. Periodic **disparate impact computation** on the window.
3. **Mock Prometheus metric emission** for observability.
4. **Alert firing** when severity reaches HIGH or CRITICAL.


In [ ]:
# ============================================================================
# Cell 18: BiasMonitoringPipeline — Streaming Simulation
# Chapter 12, p.17–19 (Bias Monitoring Pipeline)
# Author: Imran Ahmad
# ============================================================================

pipeline = BiasMonitoringPipeline(window_size=50, alert_threshold=0.80)

# Stream records from the HR dataset into the monitoring window
logger.info("Streaming HR decisions into monitoring pipeline...")
for idx, row in hr_df.iterrows():
    pipeline.ingest({
        "gender": row["gender"],
        "qualified": row["qualified"],
        "raw_score": row["raw_score"],
    })

    # Evaluate every 50 records
    if (idx + 1) % 50 == 0:
        result = pipeline.evaluate(group_col="gender", outcome_col="qualified")
        if result.get("status") != "insufficient_data":
            mp = result["metric_point"]
            logger.info(
                f"  Window {idx + 1}: DI={mp['disparate_impact_min']:.2f}, "
                f"Severity={mp['severity']}, Alert={'YES' if result['alert_fired'] else 'no'}"
            )

# Show alert history
alerts = pipeline.get_alerts()
logger.info(f"\n=== Alert History ({len(alerts)} alert(s)) ===")
for alert in alerts:
    logger.error(f"  [{alert['timestamp'][:19]}] {alert['message']}")

# Plot metric history
history = pipeline.get_metric_history()
if history:
    fig, ax = plt.subplots(figsize=(8, 3))
    di_values = [h["disparate_impact_min"] for h in history if h["disparate_impact_min"] is not None]
    ax.plot(range(len(di_values)), di_values, "o-", color="#4C72B0", label="Disparate Impact")
    ax.axhline(y=0.80, color="red", linestyle="--", label="Four-fifths threshold")
    ax.set_xlabel("Evaluation Window")
    ax.set_ylabel("Disparate Impact Ratio")
    ax.set_title("Bias Monitoring: Disparate Impact Over Time (p.17-19)")
    ax.legend()
    plt.tight_layout()
    plt.show()


## Fair Hiring Agent — Case Study (p.20–23)

The `FairHiringAgent` implements a three-layer pipeline:

1. **Anonymize** — Strip protected attributes (gender, ethnicity, institution) to prevent
   direct discrimination (p.21).
2. **Evaluate** — Score candidates on skills and experience using the chapter's
   deterministic formula (p.20-21).
3. **Bias Check + Mitigate** — Run `BiasDetector`, and if a four-fifths violation
   is found, apply `FairnessEnforcer` to correct scores (p.22-23).

Three mitigation strategies are available:
- **Reweighting** — boost scores for disadvantaged groups proportionally.
- **Threshold adjustment** — lower the passing bar for disadvantaged groups.
- **Representation learning** — project scores into a group-invariant space.


In [ ]:
# ============================================================================
# Cell 20: FairHiringAgent — Anonymization Layer
# Chapter 12, p.21 (Anonymization)
# Author: Imran Ahmad
# ============================================================================

fair_agent = FairHiringAgent()

# Show anonymization on a sample candidate
sample = hr_df.iloc[0].to_dict()
logger.info("=== Before Anonymization ===")
for k, v in sample.items():
    if k != "skills":
        logger.info(f"  {k}: {v}")
    else:
        logger.info(f"  {k}: {v[:3]}...")

anonymized = fair_agent.anonymize(sample)
logger.info("\n=== After Anonymization ===")
for k, v in anonymized.items():
    if k != "skills":
        logger.info(f"  {k}: {v}")
    else:
        logger.info(f"  {k}: {v[:3]}...")

removed = set(sample.keys()) - set(anonymized.keys())
logger.success(f"\nRemoved fields: {removed}")


In [ ]:
# ============================================================================
# Cell 21: FairHiringAgent — Full Pipeline with Mitigation
# Chapter 12, p.20–23 (Fair Hiring Pipeline)
# Author: Imran Ahmad
# ============================================================================

logger.info("Running FairHiringAgent pipeline with reweighting mitigation...\n")
pipeline_result = fair_agent.run_pipeline(hr_df, mitigation_strategy="reweighting")

# Before metrics
before_di = pipeline_result["before"]["metrics"]["disparate_impact"]
logger.info("=== BEFORE Mitigation ===")
logger.info(f"  Disparate impact ratios: {before_di['impact_ratios']}")
logger.info(f"  Min ratio: {before_di['min_ratio']:.4f}")
logger.info(f"  Four-fifths violation: {before_di['four_fifths_violation']}")
logger.info(f"  Severity: {pipeline_result['before']['severity']}")

# After metrics
after_di = pipeline_result["after"]["metrics"]["disparate_impact"]
logger.info("\n=== AFTER Mitigation ===")
logger.info(f"  Disparate impact ratios: {after_di['impact_ratios']}")
logger.info(f"  Min ratio: {after_di['min_ratio']:.4f}")
logger.info(f"  Four-fifths violation: {after_di['four_fifths_violation']}")
logger.info(f"  Severity: {pipeline_result['after']['severity']}")
logger.info(f"  Strategy used: {pipeline_result.get('strategy', 'N/A')}")


In [ ]:
# ============================================================================
# Cell 22: Before/After Fairness Comparison — Visual
# Chapter 12, p.22–23 (Mitigation Results)
# Author: Imran Ahmad
# ============================================================================

before_di = pipeline_result["before"]["metrics"]["disparate_impact"]
after_di = pipeline_result["after"]["metrics"]["disparate_impact"]

groups = list(before_di["impact_ratios"].keys())
before_vals = [before_di["impact_ratios"][g] for g in groups]
after_vals = [after_di["impact_ratios"][g] for g in groups]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Before
colors_before = ["#FF6B6B" if v < 0.80 else "#51CF66" for v in before_vals]
axes[0].barh(groups, before_vals, color=colors_before)
axes[0].axvline(x=0.80, color="red", linestyle="--", linewidth=2)
axes[0].set_xlim(0, 1.1)
axes[0].set_title("BEFORE Mitigation", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Disparate Impact Ratio")
for i, v in enumerate(before_vals):
    axes[0].text(v + 0.02, i, f"{v:.2f}", va="center", fontweight="bold")

# After
colors_after = ["#FF6B6B" if v < 0.80 else "#51CF66" for v in after_vals]
axes[1].barh(groups, after_vals, color=colors_after)
axes[1].axvline(x=0.80, color="red", linestyle="--", linewidth=2)
axes[1].set_xlim(0, 1.1)
axes[1].set_title("AFTER Mitigation (Reweighting)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Disparate Impact Ratio")
for i, v in enumerate(after_vals):
    axes[1].text(v + 0.02, i, f"{v:.2f}", va="center", fontweight="bold")

plt.suptitle("FairHiringAgent: Before vs. After — Disparate Impact (p.22-23)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

logger.success(
    f"Mitigation improved min DI from {before_di['min_ratio']:.2f} "
    f"to {after_di['min_ratio']:.2f}."
)


## Summary & Exercises

### Key Takeaways

1. **Deontic logic** (p.5–7) provides a formal framework for encoding ethical rules
   as obligations, permissions, and prohibitions — with provable consistency guarantees.

2. The **Ethical Consistency Theorem** (p.7) ensures no action can be simultaneously
   obligatory and forbidden. Violations halt the system and trigger escalation.

3. The **EU AI Act** (p.10–11) mandates seven requirements. Compliance checking can
   be automated but requires human oversight as the ultimate backstop.

4. The **Impossibility Theorem** (p.12–13) proves that no single model can satisfy
   demographic parity, equal opportunity, and predictive parity simultaneously when
   base rates differ. Organizations must choose their fairness priority.

5. **Bias detection** (p.14–19) requires continuous monitoring, not just training-time
   audits. The four-fifths rule (disparate impact ratio ≥ 0.80) is the standard threshold.

6. **Fair hiring** (p.20–23) combines anonymization, merit-based scoring, and post-hoc
   bias mitigation. The three-layer architecture ensures fairness without sacrificing utility.

### Suggested Exercises

- **Exercise 1:** Add a new violation category (e.g., "age_discrimination") to the
  `EthicalReasoningAgent` validators and test it.
- **Exercise 2:** Modify `generate_hr_dataset()` to inject ethnicity-based bias and
  measure disparate impact across ethnicity groups.
- **Exercise 3:** Compare all three `FairnessEnforcer` strategies (reweighting,
  threshold adjustment, representation learning) on the same dataset and plot
  the results side by side.
- **Exercise 4:** Implement a custom fairness metric (e.g., predictive parity) and
  add it to the `BiasDetector`.

---

*Author: Imran Ahmad — Packt Publishing*
*Chapter 12: Ethical and Explainable Agents (p.3–23)*
